# День 5 — Первая регрессия

## Цель
Понять отличие **регрессии** от классификации и обучить модель, которая предсказывает **число**.

## Регрессия vs классификация

| | Классификация (day 4) | Регрессия (day 5) |
|---|----------------------|-------------------|
| **y** | класс (0/1/2, A/B/C) | число |
| **predict** | категория | непрерывное значение |
| **Метрики** | accuracy, precision, recall, F1 | MAE, RMSE, R² |
| **Baseline** | `DummyClassifier(most_frequent)` | `DummyRegressor(mean)` |

Интерфейс тот же: **`fit(X_train, y_train)`** → **`predict(X_test)`**.

## Метрики регрессии

| Аббревиатура | Расшифровка | Когда лучше использовать | Пример |
|--------------|-------------|--------------------------|--------|
| **MAE** | **M**ean **A**bsolute **E**rror | Когда **все ошибки одинаково важны** и нужно просто объяснить результат в единицах y | Прогноз цены: MAE = 5000 → «в среднем ошибаемся на ±5000». Расчёт: `y_true=[100,200,300]`, `y_pred=[110,190,280]` → MAE = **13.3** |
| **MSE** | **M**ean **S**quared **E**rror | Когда **большие промахи особенно плохи**; редко смотрят отдельно — чаще берут RMSE | **Прогноз времени доставки (мин):** `y_true=[30,30,30]`. Модель A: `[31,31,29]` (ошибки 1,1,1). Модель B: `[30,30,33]` (ошибки 0,0,3). MAE у обеих = **1**, но MSE у B = **3** (хуже) — один большой промах «наказывается» сильнее |
| **RMSE** | **R**oot **M**SE | Как MSE (штраф за большие ошибки), но **в тех же единицах, что y**; частый стандарт в ML и соревнованиях | Прогноз дозы лекарства: ошибка 100 мг опаснее десяти по 10 мг. Расчёт: RMSE = √200 ≈ **14.1** (> MAE из‑за ошибки 20) |
| **R²** | coefficient of determination | Когда нужно **сравнить модели между собой** или понять «насколько лучше, чем просто среднее» | `y_pred` идеально → **R² = 1.0**; всегда среднее → **R² = 0.0**; хуже среднего → **R² < 0** |

**Кратко:**
- объяснить заказчику «на сколько ошибаемся» → **MAE**
- большие ошибки критичны → **RMSE** (или MSE)
- сравнить качество моделей на одном датасете → **R²**

В sklearn: `mean_absolute_error` → MAE, `mean_squared_error` → MSE, `r2_score` → R². RMSE: `MSE ** 0.5`.

Чем **ниже** MAE и RMSE — тем лучше. Чем **выше** R² — тем лучше.

## График y_true vs y_pred

Scatter-график «истинное y» против «предсказанного y»:

- точки близко к диагонали → прогнозы хорошие;
- разброс вокруг диагонали → модель ошибается;
- систематический сдвиг → модель завышает или занижает.

Полезно смотреть **для лучшей модели** — цифры MAE/RMSE не показывают форму ошибки.

## Baseline для регрессии

**`DummyRegressor(strategy="mean")`** — всегда предсказывает среднее `y_train`.

Как в day 3–4: модель **обязана** побить baseline. Если MAE модели ≈ MAE baseline и R² ≈ 0 — признаки не используются.

## Задания

1. Загрузить `load_diabetes()`, собрать X/y, сделать train/test split.
2. Обучить `DummyRegressor` (baseline) → MAE, RMSE, R² на test.
3. Обучить `LinearRegression` → MAE, RMSE, R² на test.
4. Обучить `DecisionTreeRegressor` → MAE, RMSE, R² на test.
5. Сравнить все модели в одной таблице (baseline vs model 1 vs model 2).
6. Построить график y_true vs y_pred для лучшей модели → `week-3/figures/`.
7. Написать 6–8 строк выводов: насколько ошибка большая, что можно улучшать.


In [ ]:
# 1. Загрузка Diabetes и train/test split

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

dataset = load_diabetes(as_frame=True)
X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Признаки:", list(X.columns))
print("y (первые 5):", y.head().tolist())
print("=====================================================")
print(X.head(3))
print(y.head(3))

In [ ]:
# 2. Baseline — DummyRegressor(strategy="mean")

from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

y_pred_baseline = baseline.predict(X_test)

mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
rmse_baseline = mean_squared_error(y_test, y_pred_baseline) ** 0.5
r2_baseline = r2_score(y_test, y_pred_baseline)

print(f"Baseline MAE:  {mae_baseline:.2f}")
print(f"Baseline RMSE: {rmse_baseline:.2f}")
print(f"Baseline R²:   {r2_baseline:.3f}")
print(f"Среднее y_train (что предсказывает baseline): {y_train.mean():.2f}")

In [ ]:
# 3. LinearRegression

from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = mean_squared_error(y_test, y_pred_lr) ** 0.5
r2_lr = r2_score(y_test, y_pred_lr)

print(f"LinearRegression MAE:  {mae_lr:.2f}")
print(f"LinearRegression RMSE: {rmse_lr:.2f}")
print(f"LinearRegression R²:   {r2_lr:.3f}")
print(f"Улучшение MAE vs baseline: {mae_baseline - mae_lr:.2f}")

In [ ]:
# 4. DecisionTreeRegressor

from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

mae_dt = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = mean_squared_error(y_test, y_pred_dt) ** 0.5
r2_dt = r2_score(y_test, y_pred_dt)

print(f"DecisionTree MAE:  {mae_dt:.2f}")
print(f"DecisionTree RMSE: {rmse_dt:.2f}")
print(f"DecisionTree R²:   {r2_dt:.3f}")
print(f"vs baseline (MAE): {mae_baseline - mae_dt:.2f}")
print(f"vs LinearRegression (MAE): {mae_lr - mae_dt:.2f}")

In [ ]:
# 5. Сравнение моделей: baseline vs LinearRegression vs DecisionTree

import pandas as pd

comparison = pd.DataFrame([
    {"model": "Baseline (DummyRegressor)", "MAE": mae_baseline, "RMSE": rmse_baseline, "R2": r2_baseline},
    {"model": "LinearRegression", "MAE": mae_lr, "RMSE": rmse_lr, "R2": r2_lr},
    {"model": "DecisionTreeRegressor", "MAE": mae_dt, "RMSE": rmse_dt, "R2": r2_dt},
])

comparison = comparison.sort_values(by="MAE", ascending=True).reset_index(drop=True)
comparison

In [ ]:
# 6. y_true vs y_pred для лучшей модели (LinearRegression)

from pathlib import Path
import matplotlib.pyplot as plt

figures_dir = Path("week-3/figures")
if not figures_dir.exists():
    figures_dir = Path("../figures")  # если запуск из week-3/notebooks/
figures_dir.mkdir(parents=True, exist_ok=True)

fig_path = figures_dir / "05_y_true_vs_y_pred_linear_regression.png"

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_lr, alpha=0.7, edgecolor="k")

min_val = min(y_test.min(), y_pred_lr.min())
max_val = max(y_test.max(), y_pred_lr.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2)

plt.title("Day 5: y_true vs y_pred (LinearRegression)")
plt.xlabel("y_true")
plt.ylabel("y_pred")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(fig_path, dpi=150)
plt.show()

print(f"График сохранен: {fig_path}")

## Выводы (6–8 строк)

_Заполни после заданий 1–6:_

1. Какая модель лучше по MAE / RMSE / R²?
2. Насколько модели бьют baseline?
3. Насколько велика ошибка в контексте данных?
4. Что показывает график y_true vs y_pred?
5. Можно ли доверять результату на test?
6. Что можно улучшить дальше?


---

# Day 5 — First Regression

## Goal
Understand how **regression** differs from classification and train a model that predicts a **number**.

## Regression vs classification

| | Classification (day 4) | Regression (day 5) |
|---|------------------------|---------------------|
| **y** | class | number |
| **predict** | category | continuous value |
| **Metrics** | accuracy, precision, recall, F1 | MAE, RMSE, R² |
| **Baseline** | `DummyClassifier(most_frequent)` | `DummyRegressor(mean)` |

Same interface: **`fit(X_train, y_train)`** → **`predict(X_test)`**.

## Regression metrics

| Abbrev. | Full name | When to use | Example |
|---------|-----------|-------------|---------|
| **MAE** | **M**ean **A**bsolute **E**rror | When **all errors matter equally** and you need an easy explanation in y units | Price forecast: MAE = 5000 → «on average ±5000 off». Calc: `y_true=[100,200,300]`, `y_pred=[110,190,280]` → MAE = **13.3** |
| **MSE** | **M**ean **S**quared **E**rror | When **large misses are especially bad**; rarely used alone — usually take RMSE | **Delivery time forecast (min):** `y_true=[30,30,30]`. Model A: `[31,31,29]` (errors 1,1,1). Model B: `[30,30,33]` (errors 0,0,3). Same MAE = **1**, but MSE for B = **3** (worse) — one large miss is penalized harder |
| **RMSE** | **R**oot MSE | Like MSE (penalizes large errors) but **in the same units as y**; common ML benchmark | Drug dose: an error of 100 mg is worse than ten errors of 10 mg. Calc: RMSE = √200 ≈ **14.1** (> MAE because of the error of 20) |
| **R²** | coefficient of determination | When you need to **compare models on the same dataset** or ask «how much better than the mean?» | perfect `y_pred` → **R² = 1.0**; always the mean → **R² = 0.0**; worse than mean → **R² < 0** |

**Quick guide:**
- explain to stakeholders «how far off» → **MAE**
- large errors are critical → **RMSE** (or MSE)
- compare model quality on one dataset → **R²**

In sklearn: `mean_absolute_error` → MAE, `mean_squared_error` → MSE, `r2_score` → R². RMSE: `MSE ** 0.5`.

Lower MAE/RMSE is better. Higher R² is better.

## y_true vs y_pred plot

Scatter of true y vs predicted y:

- points near the diagonal → good predictions;
- spread around the diagonal → errors;
- systematic shift → model over- or under-predicts.

Useful for the **best model** — MAE/RMSE alone do not show error shape.

## Baseline for regression

**`DummyRegressor(strategy="mean")`** — always predicts mean of `y_train`.

As in days 3–4: the model **must** beat baseline.

## Tasks (do by hand)

1. Load `load_diabetes()`, build X/y, train/test split.
2. Train `DummyRegressor` (baseline) → MAE, RMSE, R² on test.
3. Train `LinearRegression` → MAE, RMSE, R² on test.
4. Train `DecisionTreeRegressor` → MAE, RMSE, R² on test.
5. Compare all models in one table (baseline vs model 1 vs model 2).
6. Plot y_true vs y_pred for the best model → `week-3/figures/`.
7. Write 6–8 lines of conclusions: how large is the error, what to improve.


## Conclusions (6–8 lines)

_Fill in after tasks 1–6:_

1. Which model is best by MAE / RMSE / R²?
2. How much do models beat baseline?
3. How large is the error in context?
4. What does the y_true vs y_pred plot show?
5. Can you trust the test result?
6. What could be improved next?
